# Piloto Text2KG — Modelos até 3B (Colab T4)

**Configuração antes de rodar:**
1. Runtime → Change runtime type → **T4 GPU**
2. No painel lateral esquerdo, clique no ícone de **chave** (🔑) → Secrets:
   - Adicione `GITHUB_TOKEN` e ligue **Notebook access**
   - Adicione `HF_TOKEN` e ligue **Notebook access**
3. Para rodar: Runtime → **Run all**

**Modelos suportados (escolha na Célula 1):**
- `Qwen/Qwen2.5-3B-Instruct` — 3B, bfloat16 (padrão)
- `Qwen/Qwen2.5-1.5B-Instruct` — 1.5B, mais rápido
- `Qwen/Qwen2.5-3B-Instruct-AWQ` — 3B AWQ, se disponível

**Tempo estimado:** ~1.5h para 100 frases × 2 ontologias × 3 condições.

In [ ]:
# Célula 1 — Configuração
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
HF_TOKEN     = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# --- Escolha o modelo ---
MODELO   = "Qwen/Qwen2.5-3B-Instruct"   # altere aqui
DTYPE    = "bfloat16"                     # T4 aceita bf16; use 'half' se der erro
PORT     = 8000
N_FRASES = 100   # Colab T4 single: 100 frases cabe no tempo da sessão
SEED     = 42

REPO_URL     = f"https://{GITHUB_TOKEN}@github.com/felipemoraissoares/does-ontology-matter.git"
BENCH_URL    = "https://github.com/cenguix/Text2KGBench.git"
BENCH_COMMIT = "50a3d255371b8817cdff70fd88459ac82b339cfe"

REPO_DIR   = "/content/does-ontology-matter"
BENCH_DIR  = "/content/Text2KGBench"
RESULT_DIR = f"{REPO_DIR}/resultados/colab"
C2_DIR     = f"{REPO_DIR}/configs/c2"
VLLM_LOG   = "/content/vllm.log"

GIT_EMAIL = "felipemoraissoares@gmail.com"
GIT_NAME  = "Felipe Morais (Colab)"

print(f"Modelo:       {MODELO}")
print(f"N_frases:     {N_FRASES} por experimento")
print(f"BENCH_COMMIT: {BENCH_COMMIT[:12]}...")

In [ ]:
# Célula 2 — Helpers
import subprocess

def run(cmd, check=True, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Falhou (rc={result.returncode}): {cmd[:120]}")
    return result

def git(cmd):
    return run(f'git -C "{REPO_DIR}" {cmd}', check=False)

In [ ]:
# Célula 3 — Clonar repositórios
if not os.path.exists(REPO_DIR):
    run(f"git clone --depth=1 {REPO_URL} {REPO_DIR}")
else:
    git("pull")

# Clona Text2KGBench no commit fixo para reprodutibilidade
if not os.path.exists(BENCH_DIR):
    run(f"git init {BENCH_DIR}")
    run(f"git -C {BENCH_DIR} fetch --depth=1 {BENCH_URL} {BENCH_COMMIT}")
    run(f"git -C {BENCH_DIR} checkout FETCH_HEAD")
    actual = run(f"git -C {BENCH_DIR} rev-parse HEAD", check=False).stdout.strip()
    if actual != BENCH_COMMIT:
        print(f"AVISO: bench commit {actual[:12]} difere do esperado {BENCH_COMMIT[:12]}")
    else:
        print(f"Text2KGBench OK — commit {BENCH_COMMIT[:12]}")
else:
    print("Text2KGBench já existe.")

git(f"config user.email '{GIT_EMAIL}'")
git(f"config user.name '{GIT_NAME}'")
print("Clones OK.")

In [ ]:
# Célula 4 — Instalar dependências
run("pip install -q vllm openai sentence-transformers scikit-learn nltk scipy numpy pandas statsmodels")

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(C2_DIR, exist_ok=True)
print("Dependências OK.")

In [ ]:
# Célula 5 — Subir vLLM em segundo plano
# T4 single: tensor-parallel-size=1

vllm_cmd = (
    f"vllm serve {MODELO} "
    f"--port {PORT} "
    f"--dtype {DTYPE} "
    f"--tensor-parallel-size 1 "
    f"--max-model-len 4096 "
    f"--gpu-memory-utilization 0.90 "
    f"> {VLLM_LOG} 2>&1 &"
)
os.system(vllm_cmd)
print(f"vLLM iniciado (modelo={MODELO}).")

In [ ]:
# Célula 6 — Aguardar vLLM pronto (polling automático)
import time, urllib.request, urllib.error

VLLM_URL = f"http://localhost:{PORT}"
TIMEOUT  = 240
INTERVALO = 10
elapsed   = 0

while elapsed < TIMEOUT:
    try:
        urllib.request.urlopen(f"{VLLM_URL}/health", timeout=3)
        print(f"vLLM pronto após {elapsed}s.")
        break
    except (urllib.error.URLError, ConnectionRefusedError, OSError):
        if elapsed % 30 == 0:
            print(f"  aguardando vLLM... ({elapsed}s)")
        time.sleep(INTERVALO)
        elapsed += INTERVALO
else:
    print(f"ERRO: vLLM não respondeu em {TIMEOUT}s. Log:")
    run(f"tail -40 {VLLM_LOG}", check=False)
    raise RuntimeError("vLLM não iniciou.")

In [ ]:
# Célula 7 — Gerar manifesto do piloto com caminhos do Colab
import json

modelo_slug = MODELO.split("/")[-1].lower()

run(
    f"python {REPO_DIR}/scripts/gerar_manifesto.py "
    f"--modelo {MODELO} "
    f"--url http://localhost:{PORT}/v1 "
    f"--saida_dir {REPO_DIR}/configs/pipeline",
    cwd=REPO_DIR
)

piloto_src = f"{REPO_DIR}/configs/pipeline/manifesto_piloto_kaggle.json"
with open(piloto_src, encoding="utf-8") as f:
    piloto = json.load(f)

for exp in piloto:
    exp["saida"]       = f"{RESULT_DIR}/{os.path.basename(exp['saida'])}"
    exp["frases_max"]  = N_FRASES
    exp["frases_seed"] = SEED

piloto_path = f"/content/manifesto_piloto_colab.json"
with open(piloto_path, "w", encoding="utf-8") as f:
    json.dump(piloto, f, ensure_ascii=False, indent=2)

print(f"Manifesto com {len(piloto)} experimentos -> {piloto_path}")

In [ ]:
# Célula 8 — Executar piloto com push automático

PYPATH = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}

run(
    f"python -m t2kg.run "
    f"--manifesto {piloto_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--skip_existentes "
    f"--auto_push "
    f"--push_repo_dir \"{REPO_DIR}\" "
    f"--verbose",
    cwd=REPO_DIR,
    env=PYPATH
)
print("\nPiloto concluído.")

In [ ]:
# Célula 9 — Indução C2 + calibração (opcional — pode não caber no tempo do Colab)

for onto in ["1_movie"]:
    calib_file = f"{C2_DIR}/calibracao_tekgen_{onto}.json"
    if os.path.exists(calib_file):
        print(f"\n--- tekgen/{onto}: calibracao ja existe, pulando ---")
        continue
    print(f"\n--- Induzindo C2 para tekgen/{onto} ---")
    run(
        f"python {REPO_DIR}/scripts/induzir_c2.py "
        f"--dataset tekgen "
        f"--onto {onto} "
        f"--url http://localhost:{PORT}/v1 "
        f"--modelo_llm {MODELO} "
        f"--threshold 0.80 --n_frases 50 --seed 42 --calibrar "
        f"--saida_dir {C2_DIR}",
        cwd=REPO_DIR,
        env=PYPATH
    )

print("Indução C2 concluída.")

In [ ]:
# Célula 10 — Avaliar resultados

CSV_OUT = f"{RESULT_DIR}/tabela_colab.csv"

run(
    f"python {REPO_DIR}/scripts/avaliar_resultados.py "
    f"--manifesto {piloto_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--csv {CSV_OUT}",
    cwd=REPO_DIR,
    env=PYPATH
)

import pandas as pd
df = pd.read_csv(CSV_OUT)
print(df.to_string())

In [ ]:
# Célula 11 — Push final ou download local
# Opção A: push ao GitHub
git("add -A")
git(f'commit -m "colab: {modelo_slug}, {N_FRASES} frases, C0/C1/C1_oneshot"')
git("push")
print("Push OK.")

# Opção B: download local (descomente se preferir)
# from google.colab import files
# import glob
# for f in glob.glob(f"{RESULT_DIR}/*.jsonl") + glob.glob(f"{RESULT_DIR}/*.csv") + glob.glob(f"{C2_DIR}/*.json"):
#     files.download(f)